# Toy DGD: Two MNIST Digits, 2D Latent, 2-Component GMM

The other toy notebook in this folder (`toy_dgd_blobs.ipynb`) uses synthetic data and a small MLP decoder -- useful for isolating the training mechanics, but it turned out the synthetic data's within-cluster variation was easy to over- or under-structure by hand, and a plain MLP is a different architecture from the one the main pipeline actually uses. This notebook instead trains on two real, visually very different MNIST digits (`0` and `1`), using **the exact same `ConvDecoder` class and `config.yaml` decoder settings as the FashionMNIST pipeline** -- so a good result here is evidence the architecture itself works, not just evidence about a hand-designed toy distribution.

Everything else mirrors `toy_dgd_blobs.ipynb` and `config/config.yaml`: zero-init representations, noise injection, separate decoder/train-rep/val-rep optimizers with cosine LR schedules, an 80/10/10 train/val/test split, a periodically-refit GMM prior, and a held-out inference pass mirroring `dgd_test_inference.ipynb`'s Algorithm 2. Runs on CPU in a couple of minutes -- 800 training images at 28x28 is still tiny by deep learning standards.

## The math

**Data**: $N$ MNIST images of the digits $0$ and $1$ ($x_i \in [0,1]^{1 \times 28 \times 28}$, label $y_i$ never seen by the model), split 80/10/10 into train/val/test exactly like `config.yaml`'s `data.val_split: 0.1`, `data.test_split: 0.1`. Unlike the synthetic blobs notebook, there's no generative formula for $x_i$ here -- it's real handwritten digit data, with genuinely correlated, structured within-class variation (stroke slant, thickness, size) that a synthetic i.i.d.-noise model can't easily reproduce.

**Model** -- decoder $f_\theta: \mathbb{R}^2 \to [0,1]^{1 \times 28 \times 28}$, `ConvDecoder` with `config.yaml`'s exact decoder settings (`hidden_dims=[128,64]`, `output_size=(28,28)`, `init_size=(7,7)`, `final_activation="sigmoid"`, batch norm, nearest-neighbor upsampling) -- and free per-sample latents $z_i \in \mathbb{R}^2$ (one set for train, a separate set for val), all initialized at exactly $\mathbf{0}$ (`distribution: "zeros"`), regularized by a $K{=}2$-component Gaussian-mixture prior fit only to the *train* latents:

$$
\mathcal{L}(\theta, Z) = \sum_{i=1}^N \|f_\theta(\tilde z_i) - x_i\|_2^2 \;-\; \lambda \sum_{i=1}^N \log p_{\text{GMM}}(\tilde z_i), \qquad p_{\text{GMM}}(z) = \sum_{c=1}^{2} \pi_c\, \mathcal{N}(z; \mu_c, \sigma_c^2 I)
$$

where $\tilde z_i = z_i + \epsilon_i$, $\epsilon_i \sim \mathcal{N}(0, \sigma_t^2 I)$, annealed from $\sigma_t{=}1.0$ down to $0.01$ over training -- same mechanism as `noise_injection_explained.ipynb` and the blobs notebook, verified below both numerically (realized displacement vs. the theoretical $\sigma\sqrt{\pi/2}$ expectation) and visually (a displacement line from each point's clean $z$ to its actual noised $\tilde z$, in every animation's latent panel).

**Optimization** -- block-coordinate, matching `DGDTrainer`: separate AdamW optimizers for $\theta$, $Z_{\text{train}}$, and $Z_{\text{val}}$, each with its own cosine-annealed learning rate. Every epoch: a full train step (decoder + train latents), then a val step with the decoder's gradients disabled so only $Z_{\text{val}}$ moves. A reconstruction-only warm-up precedes the GMM term, and the GMM is periodically refit via EM to the current (clean, un-noised) train $Z$ only. Both loss terms use `reduction='sum'` for backprop; the loss curves plotted below show the mean-per-sample value instead, matching `trainer.py`'s own tracking convention.

In [ ]:
import sys
import time
from pathlib import Path
from datetime import timedelta
import io
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.collections import LineCollection
from PIL import Image
from sklearn.decomposition import PCA
from torchvision import datasets, transforms

current_dir = Path.cwd()
project_root = current_dir.parent if 'notebooks' in current_dir.parts else current_dir
sys.path.append(str(project_root))
sys.path.append(str(project_root / 'src'))

from src.models import RepresentationLayer, ConvDecoder
from src.utils.schedules import cosine_noise_schedule
from tgmm import GaussianMixture, ClusteringMetrics
from tgmm.plotting import plot_gmm

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cpu')  # tiny problem, no need for a GPU

In [ ]:
digit_a, digit_b = 0, 1   # visually very different: a closed loop vs. a single stroke
n_per_digit = 500
N_total = 2 * n_per_digit

mnist_train = datasets.MNIST(root=str(project_root / 'data'), train=True, download=False, transform=transforms.ToTensor())
targets = mnist_train.targets
idx_a = torch.where(targets == digit_a)[0][:n_per_digit]
idx_b = torch.where(targets == digit_b)[0][:n_per_digit]
sel = torch.cat([idx_a, idx_b])

x_all = torch.stack([mnist_train[i][0] for i in sel])  # [N, 1, 28, 28], already in [0, 1] via ToTensor
y_all = torch.cat([torch.zeros(n_per_digit, dtype=torch.long), torch.ones(n_per_digit, dtype=torch.long)])

perm = torch.randperm(N_total)
x_all, y_all = x_all[perm], y_all[perm]

# 80/10/10 split, same ratios as config.yaml's data.val_split=0.1, data.test_split=0.1
n_train = int(0.8 * N_total)
n_val = int(0.1 * N_total)
n_test = N_total - n_train - n_val

x_train, y_train = x_all[:n_train], y_all[:n_train]
x_val, y_val = x_all[n_train:n_train + n_val], y_all[n_train:n_train + n_val]
x_test, y_test = x_all[n_train + n_val:], y_all[n_train + n_val:]

N, N_val, N_test = x_train.shape[0], x_val.shape[0], x_test.shape[0]
dim_x_flat = x_train.shape[1] * x_train.shape[2] * x_train.shape[3]  # 784, for PCA/MSE on flattened images

print(f"Digits {digit_a} and {digit_b}, {n_per_digit} each, 28x28 grayscale in [0, 1]")
print(f"Split 80/10/10: train={N}, val={N_val}, test={N_test} (total {N_total})")

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(12, 2.6))
for col in range(10):
    i_a = (y_train == 0).nonzero()[col].item()
    i_b = (y_train == 1).nonzero()[col].item()
    axes[0, col].imshow(x_train[i_a, 0], cmap='gray', vmin=0, vmax=1); axes[0, col].axis('off')
    axes[1, col].imshow(x_train[i_b, 0], cmap='gray', vmin=0, vmax=1); axes[1, col].axis('off')
axes[0, 0].set_title(f'digit {digit_a}', loc='left', fontsize=9)
axes[1, 0].set_title(f'digit {digit_b}', loc='left', fontsize=9)
fig.suptitle('A few training examples per class')
plt.tight_layout()
plt.show()

# PCA fit once, on the flattened training split only -- reused everywhere below
# (val, test, every reconstruction panel), exactly like the blobs notebook.
pca_raw = PCA(n_components=2, random_state=42)
x_train_pca = pca_raw.fit_transform(x_train.reshape(N, -1).numpy())
x_val_pca = pca_raw.transform(x_val.reshape(N_val, -1).numpy())
x_test_pca = pca_raw.transform(x_test.reshape(N_test, -1).numpy())

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(x_train_pca[:, 0], x_train_pca[:, 1], c=y_train.numpy(), cmap='coolwarm', s=12, alpha=0.7)
ax.set_title(f"Raw pixel data, PCA projection ({pca_raw.explained_variance_ratio_.sum()*100:.1f}% variance explained)")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
plt.tight_layout()
plt.show()

## Model and training

Deviations from `config.yaml`, all just complexity/scale, not mechanism -- notably the **decoder itself has zero deviations** this time:

| | `config.yaml` (FashionMNIST) | here |
|---|---|---|
| `representation.n_features` | 8 | 2 (kept small on purpose -- the point of this notebook) |
| `decoder.*` (`hidden_dims`, `output_size`, `init_size`, `final_activation`, everything) | -- | **identical**, same `ConvDecoder` class |
| `gmm.n_components` | 20 | 2 (one per digit) |
| `gmm.covariance_type` | `tied_spherical` | `spherical` (each component gets its own variance) |
| `training.epochs` | 200 | 100 |
| `training.first_epoch_gmm` / `refit_gmm_interval` | 50 / 50 | 25 / 25 |
| `data.val_split` / `data.test_split` | 0.1 / 0.1 | 0.1 / 0.1 (same -- 1000 images total instead of FashionMNIST's) |
| everything else (`distribution: "zeros"`, optimizer betas/eps/lr, `lr_scheduler.*`, `latent_noise_*`, `lambda_gmm`, GMM `tol`/`reg_covar`/`init_*`) | -- | identical values |

Also still dropped, unlike `DGDTrainer`: checkpointing, early stopping, and best-model restoration -- this notebook trains for a fixed number of epochs and reports the final-epoch model, using the held-out test set at the end (genuinely never touched during training) as its generalization check instead.

In [ ]:
dim_z = 2
epochs = 100

# ConvDecoder with config.yaml's exact decoder.* settings -- the same class and
# configuration DGDTrainer builds for FashionMNIST.
decoder = ConvDecoder(
    latent_dim=dim_z,
    hidden_dims=[128, 64],
    output_channels=1,
    output_size=(28, 28),
    init_size=(7, 7),
    kernel_size=3,
    stride=2,
    padding=1,
    output_padding=1,
    normalization='batch',
    activation='leaky_relu',
    final_activation='sigmoid',
    dropout_rate=0.0,
    upsampling_mode='nearest',
)

rep = RepresentationLayer(dim=dim_z, n_samples=N, dist='zeros', dist_params={}, device=device)
val_rep = RepresentationLayer(dim=dim_z, n_samples=N_val, dist='zeros', dist_params={}, device=device)

gmm = GaussianMixture(
    n_components=2,
    n_features=dim_z,
    covariance_type='spherical',
    max_iter=1000,
    tol=1e-4,
    reg_covar=1e-6,
    n_init=1,
    init_means='kmeans',
    init_weights='uniform',
    init_covariances='empirical',
    random_state=42,
    warm_start=True,
    device=device,
)

# Decoder, train-rep, and val-rep optimizers -- same values as config.yaml's
# training.optimizer.decoder / .representation
decoder_optimizer = torch.optim.AdamW(
    decoder.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01,
)
trainrep_optimizer = torch.optim.AdamW(
    rep.parameters(), lr=0.1, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0,
)
valrep_optimizer = torch.optim.AdamW(
    val_rep.parameters(), lr=0.1, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0,
)

# Cosine LR schedules, base_lr -> final_lr -- same values as config.yaml's
# training.lr_scheduler
decoder_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(decoder_optimizer, T_max=epochs, eta_min=0.001)
trainrep_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(trainrep_optimizer, T_max=epochs, eta_min=0.01)
valrep_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(valrep_optimizer, T_max=epochs, eta_min=0.01)

decoder_params = sum(p.numel() for p in decoder.parameters())
print(f"Decoder ({type(decoder).__name__}): {decoder_params:,} params. "
      f"Train rep: {rep.n_rep} x {rep.dim}. Val rep: {val_rep.n_rep} x {val_rep.dim}.")

In [ ]:
first_epoch_gmm = 25
refit_gmm_interval = 25
lambda_gmm = 1.0
latent_noise_start = 1.0
latent_noise_end = 0.01

cluster_metrics = ClusteringMetrics()
history = {
    'train_loss': [], 'train_recon': [], 'train_gmm': [], 'train_ami': [], 'train_ari': [],
    'val_loss': [], 'val_recon': [], 'val_gmm': [], 'val_ami': [], 'val_ari': [],
    'noise_scale': [], 'noise_realized': [],
}
epoch_times = []
frames_train = []  # per-epoch snapshots for the "watching it train" animation
frames_val = []    # per-epoch snapshots for the "watching validation" animation
start_time = time.time()

# Sanity check for the val-phase requires_grad toggle below -- if re-enabling
# decoder gradients after the val step were ever missed, the decoder would
# silently stop training and everything would still "run" with plausible-
# looking (just wrong) output.
decoder_w0 = next(decoder.parameters()).detach().clone()

for epoch in range(1, epochs + 1):
    epoch_start = time.time()

    is_gmm_refit_epoch = epoch == first_epoch_gmm or (refit_gmm_interval and epoch % refit_gmm_interval == 0)
    current_train_ami, current_train_ari = 0.0, 0.0
    current_val_ami, current_val_ari = 0.0, 0.0

    if is_gmm_refit_epoch or epoch > first_epoch_gmm:
        with torch.no_grad():
            representations = rep.z.detach()
            if is_gmm_refit_epoch:
                gmm.fit(representations, max_iter=1000 if epoch == first_epoch_gmm else 100)
            else:
                gmm.fit(representations, max_iter=100, warm_start=True)
            train_pred = gmm.predict(representations)
            current_train_ami = cluster_metrics.adjusted_mutual_info_score(y_train, train_pred)
            current_train_ari = cluster_metrics.adjusted_rand_score(y_train, train_pred)
            val_pred = gmm.predict(val_rep.z.detach())
            current_val_ami = cluster_metrics.adjusted_mutual_info_score(y_val, val_pred)
            current_val_ari = cluster_metrics.adjusted_rand_score(y_val, val_pred)

    noise_scale = cosine_noise_schedule(epoch, epochs, latent_noise_start, latent_noise_end)

    # --- Train phase: decoder + train representations ---
    decoder_optimizer.zero_grad()
    trainrep_optimizer.zero_grad()

    z_clean = rep()
    train_noise = torch.randn_like(z_clean) * noise_scale if noise_scale > 0 else torch.zeros_like(z_clean)
    z = z_clean + train_noise
    x_hat = decoder(z)
    train_recon_loss = F.mse_loss(x_hat, x_train, reduction='sum')

    if epoch >= first_epoch_gmm:
        train_gmm_loss = -lambda_gmm * gmm.score_samples(z).sum()
        train_loss = train_recon_loss + train_gmm_loss
    else:
        train_gmm_loss = torch.tensor(0.0)
        train_loss = train_recon_loss

    train_loss.backward()
    decoder_optimizer.step()
    trainrep_optimizer.step()
    decoder_scheduler.step()
    trainrep_scheduler.step()

    # --- Val phase: val representations only, decoder frozen (matches DGDTrainer.train()) ---
    for p in decoder.parameters():
        p.requires_grad_(False)
    valrep_optimizer.zero_grad()

    zv_clean = val_rep()
    val_noise = torch.randn_like(zv_clean) * noise_scale if noise_scale > 0 else torch.zeros_like(zv_clean)
    zv = zv_clean + val_noise
    xv_hat = decoder(zv)
    val_recon_loss = F.mse_loss(xv_hat, x_val, reduction='sum')

    if epoch >= first_epoch_gmm:
        val_gmm_loss = -lambda_gmm * gmm.score_samples(zv).sum()
        val_loss = val_recon_loss + val_gmm_loss
    else:
        val_gmm_loss = torch.tensor(0.0)
        val_loss = val_recon_loss

    val_loss.backward()
    valrep_optimizer.step()
    valrep_scheduler.step()

    for p in decoder.parameters():
        p.requires_grad_(True)

    history['train_loss'].append(train_loss.item() / N)
    history['train_recon'].append(train_recon_loss.item() / N)
    history['train_gmm'].append(train_gmm_loss.item() / N)
    history['train_ami'].append(current_train_ami)
    history['train_ari'].append(current_train_ari)
    history['val_loss'].append(val_loss.item() / N_val)
    history['val_recon'].append(val_recon_loss.item() / N_val)
    history['val_gmm'].append(val_gmm_loss.item() / N_val)
    history['val_ami'].append(current_val_ami)
    history['val_ari'].append(current_val_ari)
    history['noise_scale'].append(noise_scale)
    history['noise_realized'].append(train_noise.norm(dim=1).mean().item())

    # Per-epoch snapshots for the animations below. x_hat_pca flattens the
    # ConvDecoder's [B,1,28,28] output before applying the same PCA used for
    # the raw pixel data above.
    with torch.no_grad():
        gmm_means = gmm.means_.detach().cpu().clone().numpy() if epoch >= first_epoch_gmm else None
        gmm_vars = gmm.covariances_.detach().cpu().clone().numpy() if epoch >= first_epoch_gmm else None
        frames_train.append({
            'step': epoch,
            'z': z_clean.detach().clone().numpy(),
            'z_noised': z.detach().clone().numpy(),
            'x_hat_pca': pca_raw.transform(decoder(z_clean).detach().reshape(N, -1).numpy()),
            'means': gmm_means, 'vars': gmm_vars,
        })
        frames_val.append({
            'step': epoch,
            'z': zv_clean.detach().clone().numpy(),
            'z_noised': zv.detach().clone().numpy(),
            'x_hat_pca': pca_raw.transform(decoder(zv_clean).detach().reshape(N_val, -1).numpy()),
            'means': gmm_means, 'vars': gmm_vars,
        })

    epoch_duration = time.time() - epoch_start
    epoch_times.append(epoch_duration)
    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    remaining_str = str(timedelta(seconds=int((epochs - epoch) * avg_epoch_time)))

    lr_decoder = decoder_optimizer.param_groups[0]['lr']
    lr_rep = trainrep_optimizer.param_groups[0]['lr']
    train_gmm_str = f"{history['train_gmm'][-1]:.4f}" if epoch >= first_epoch_gmm else "0.0000"
    val_gmm_str = f"{history['val_gmm'][-1]:.4f}" if epoch >= first_epoch_gmm else "0.0000"
    train_ami_ari_str = f", AMI={current_train_ami:.4f}, ARI={current_train_ari:.4f}" if epoch >= first_epoch_gmm else ""
    val_ami_ari_str = f", AMI={current_val_ami:.4f}, ARI={current_val_ari:.4f}" if epoch >= first_epoch_gmm else ""

    print(f"Epoch {epoch}/{epochs} [Remaining: {remaining_str}, LR: Dec={lr_decoder:.2e}, Rep={lr_rep:.2e}, Noise={noise_scale:.4f}]")
    print(f"       - Train Loss: {history['train_loss'][-1]:.4f}, Recon: {history['train_recon'][-1]:.4f}, GMM: {train_gmm_str}{train_ami_ari_str}")
    print(f"       - Val   Loss: {history['val_loss'][-1]:.4f}, Recon: {history['val_recon'][-1]:.4f}, GMM: {val_gmm_str}{val_ami_ari_str}")

assert not torch.equal(decoder_w0, next(decoder.parameters()).detach()), "decoder did not update -- requires_grad toggle bug"

with torch.no_grad():
    gmm.fit(rep.z.detach(), max_iter=1000)

print(f"\nTraining completed in {str(timedelta(seconds=int(time.time() - start_time)))}")
print(f"Final GMM refit converged: {gmm.converged_} (iterations: {gmm.n_iter_})")
print(f"Final train loss: {history['train_loss'][-1]:.4f} (AMI={history['train_ami'][-1]:.4f}, ARI={history['train_ari'][-1]:.4f})")
print(f"Final val loss:   {history['val_loss'][-1]:.4f} (AMI={history['val_ami'][-1]:.4f}, ARI={history['val_ari'][-1]:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history['train_loss'], label='train total', color='tab:blue')
axes[0].plot(history['val_loss'], label='val total', color='tab:blue', linestyle='--')
axes[0].plot(history['train_recon'], label='train recon', color='tab:green', alpha=0.7)
axes[0].plot(history['val_recon'], label='val recon', color='tab:green', linestyle='--', alpha=0.7)
axes[0].plot(history['train_gmm'], label='train GMM', color='tab:orange', alpha=0.7)
axes[0].plot(history['val_gmm'], label='val GMM', color='tab:orange', linestyle='--', alpha=0.7)
axes[0].axvline(first_epoch_gmm, color='gray', linestyle=':', alpha=0.5, label='GMM term added')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss per sample')
axes[0].legend(fontsize=7, ncol=2)
axes[0].set_title('Training curve (train solid, val dashed)')

axes[1].plot(history['noise_scale'], color='orange')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Noise scale (sigma)')
axes[1].set_title('Noise schedule')

theoretical_noise = np.array(history['noise_scale']) * np.sqrt(np.pi / 2)
axes[2].plot(history['noise_realized'], label='realized (mean over 800 points)', color='tab:red')
axes[2].plot(theoretical_noise, label=r'theoretical $E[\|\epsilon\|]=\sigma\sqrt{\pi/2}$', color='black', linestyle=':')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Mean noise displacement')
axes[2].set_title('Noise is applied: realized vs. theoretical')
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.show()

print(f"Realized noise displacement, epoch 1: {history['noise_realized'][0]:.4f} (theoretical: {theoretical_noise[0]:.4f})")
print(f"Realized noise displacement, epoch {epochs}: {history['noise_realized'][-1]:.4f} (theoretical: {theoretical_noise[-1]:.4f})")

## Watching it train (and validate, and infer)

The same three-panel animation -- true data in PCA space (left), the latent space with a short line from each point's clean $z$ to its actual noised $\tilde z$ that step (middle), and the reconstruction in that same PCA space (right) -- gets built three times below: once for training, once for validation, once for the held-out test inference at the end. One helper function builds all three, matching the blobs notebook's conventions: axes visible, one frame for every single epoch/step (no subsampling), and GMM ellipse colors matched to the *true* cluster color they enclose (by majority vote against the trained GMM's own predictions, not just assumed from component index).

In [ ]:
_train_pred = gmm.predict(rep.z.detach()).detach().cpu().numpy()
_y_train_np = y_train.numpy()
_cmap = plt.get_cmap('coolwarm')
_label_colors = {0: _cmap(0.0), 1: _cmap(1.0)}
_n_components = gmm.means_.shape[0]
cluster_colors = []
for k in range(_n_components):
    mask = _train_pred == k
    majority_label = int(round(_y_train_np[mask].mean())) if mask.sum() > 0 else k
    cluster_colors.append(_label_colors[majority_label])

def build_three_panel_gif(
    frames, true_x_pca, labels, out_path,
    step_total, frame_stride=1, duration=250,
    left_title="True x (PCA)", mid_title="Latent z", right_title="Reconstruction (PCA)",
):
    '''Render a [true data PCA | latent z (clean + displacement line to noised) |
    reconstruction PCA] GIF. `frames` is a list of dicts with keys 'step', 'z',
    'z_noised', 'x_hat_pca', 'means' (or None), 'vars' (or None).'''
    pca_pad = 0.5
    all_recon_pca = np.concatenate([f['x_hat_pca'] for f in frames], axis=0)
    all_pca = np.concatenate([true_x_pca, all_recon_pca], axis=0)
    pca_xlim = (all_pca[:, 0].min() - pca_pad, all_pca[:, 0].max() + pca_pad)
    pca_ylim = (all_pca[:, 1].min() - pca_pad, all_pca[:, 1].max() + pca_pad)

    z_all = np.concatenate([f['z'] for f in frames], axis=0)
    z_pad = 0.5
    z_xlim = [z_all[:, 0].min() - z_pad, z_all[:, 0].max() + z_pad]
    z_ylim = [z_all[:, 1].min() - z_pad, z_all[:, 1].max() + z_pad]
    last_means = frames[-1]['means']
    if last_means is not None:
        last_stds = np.sqrt(frames[-1]['vars'])
        z_xlim[0] = min(z_xlim[0], (last_means[:, 0] - 3 * last_stds).min() - z_pad)
        z_xlim[1] = max(z_xlim[1], (last_means[:, 0] + 3 * last_stds).max() + z_pad)
        z_ylim[0] = min(z_ylim[0], (last_means[:, 1] - 3 * last_stds).min() - z_pad)
        z_ylim[1] = max(z_ylim[1], (last_means[:, 1] + 3 * last_stds).max() + z_pad)

    selected = frames[::frame_stride]
    if selected[-1]['step'] != frames[-1]['step']:
        selected.append(frames[-1])

    rng = np.random.default_rng(0)
    n_line_points = min(60, len(labels))
    line_idx = rng.choice(len(labels), size=n_line_points, replace=False)
    line_colors_base = plt.get_cmap('coolwarm')(np.asarray(labels, dtype=float)[line_idx])

    gif_frames = []
    for f in selected:
        fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.6), dpi=30)

        axes[0].scatter(true_x_pca[:, 0], true_x_pca[:, 1], c=labels, cmap='coolwarm', s=9, alpha=0.7)
        axes[0].set_xlim(pca_xlim); axes[0].set_ylim(pca_ylim)
        axes[0].set_title(left_title, fontsize=9)
        axes[0].set_xlabel("PC 1", fontsize=8); axes[0].set_ylabel("PC 2", fontsize=8)
        axes[0].tick_params(labelsize=7)

        segments = np.stack([f['z'][line_idx], f['z_noised'][line_idx]], axis=1)
        axes[1].add_collection(LineCollection(segments, colors=line_colors_base, linewidths=0.6,
                                               alpha=1.0, antialiased=False, zorder=2))
        axes[1].scatter(f['z'][:, 0], f['z'][:, 1], c=labels, cmap='coolwarm', s=8, alpha=0.9, zorder=3)
        if f['means'] is not None:
            stds = np.sqrt(f['vars'])
            for k in range(len(f['means'])):
                for n_std, alpha in zip([1, 2, 3], [0.5, 0.3, 0.15]):
                    axes[1].add_patch(Circle(f['means'][k], n_std * stds[k], facecolor=cluster_colors[k % len(cluster_colors)],
                                              edgecolor='black', linewidth=1, linestyle='--', alpha=alpha, zorder=1))
                axes[1].scatter(*f['means'][k], color='black', marker='h', s=40, zorder=4)
        axes[1].set_xlim(z_xlim); axes[1].set_ylim(z_ylim)
        status = "" if f['means'] is not None else " (GMM inactive)"
        axes[1].set_title(f"{mid_title}, step {f['step']}/{step_total}{status}", fontsize=9)
        axes[1].set_xlabel("z[0]", fontsize=8); axes[1].set_ylabel("z[1]", fontsize=8)
        axes[1].tick_params(labelsize=7)

        axes[2].scatter(f['x_hat_pca'][:, 0], f['x_hat_pca'][:, 1], c=labels, cmap='coolwarm', s=9, alpha=0.7)
        axes[2].set_xlim(pca_xlim); axes[2].set_ylim(pca_ylim)
        axes[2].set_title(right_title, fontsize=9)
        axes[2].set_xlabel("PC 1", fontsize=8); axes[2].set_ylabel("PC 2", fontsize=8)
        axes[2].tick_params(labelsize=7)

        fig.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format='png')
        plt.close(fig)
        buf.seek(0)
        gif_frames.append(Image.open(buf).convert('RGB'))

    palette_frame = gif_frames[len(gif_frames) // 4].convert('P', palette=Image.ADAPTIVE, colors=28)
    gif_frames_p = [im.quantize(palette=palette_frame, dither=Image.NONE) for im in gif_frames]

    out_path = Path(out_path)
    gif_frames_p[0].save(
        out_path, format='GIF', save_all=True, append_images=gif_frames_p[1:],
        duration=duration, loop=0, optimize=True,
    )
    print(f"Saved {len(gif_frames_p)}-frame animation to {out_path.resolve()} ({out_path.stat().st_size / 1024:.0f} KB)")

In [ ]:
build_three_panel_gif(
    frames_train, x_train_pca, y_train.numpy(), 'toy_dgd_mnist_training.gif',
    step_total=epochs, frame_stride=1, duration=250,
    left_title="True x_train (PCA)", mid_title="Latent z_train", right_title="Reconstruction (PCA)",
)

![Training animation: true train digits (left), latent z_train with a displacement line per point (middle), reconstruction (right)](toy_dgd_mnist_training.gif)

In [ ]:
build_three_panel_gif(
    frames_val, x_val_pca, y_val.numpy(), 'toy_dgd_mnist_validation.gif',
    step_total=epochs, frame_stride=1, duration=250,
    left_title="True x_val (PCA)", mid_title="Latent z_val", right_title="Reconstruction (PCA)",
)

![Validation animation: true val digits (left), latent z_val with a displacement line per point (middle), reconstruction (right)](toy_dgd_mnist_validation.gif)

Same three panels, but for the held-out validation split -- $Z_{\text{val}}$ never touches the decoder's gradients (see the training loop above), it only ever adapts to a decoder and GMM that train is shaping.

## The learned latent space

$z$ is already 2D, so this *is* the latent space -- no PCA/UMAP projection needed. Train and val latents shown side by side against the same (train-fit, frozen) GMM.

In [ ]:
z_final = rep().detach()
z_val_final = val_rep().detach()

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
plot_gmm(
    z_final, gmm=gmm,
    color_by_cluster=True, true_labels=y_train, match_labels_to_true=True,
    show_ellipses=True, ellipse_std_devs=[1, 2, 3],
    title="Train latents + GMM", xlabel="z[0]", ylabel="z[1]", ax=axes[0],
)
plot_gmm(
    z_val_final, gmm=gmm,
    color_by_cluster=True, true_labels=y_val, match_labels_to_true=True,
    show_ellipses=True, ellipse_std_devs=[1, 2, 3],
    title="Val latents + (same, frozen) GMM", xlabel="z[0]", ylabel="z[1]", ax=axes[1],
)
plt.tight_layout()
plt.show()

In [ ]:
z_pred = gmm.predict(z_final)
ami = cluster_metrics.adjusted_mutual_info_score(y_train, z_pred)
ari = cluster_metrics.adjusted_rand_score(y_train, z_pred)
print(f"Train latents vs. true digit labels: AMI={ami:.4f}, ARI={ari:.4f} (1.0 = perfect recovery)")

z_val_pred = gmm.predict(z_val_final)
val_ami = cluster_metrics.adjusted_mutual_info_score(y_val, z_val_pred)
val_ari = cluster_metrics.adjusted_rand_score(y_val, z_val_pred)
print(f"Val latents vs. true digit labels:   AMI={val_ami:.4f}, ARI={val_ari:.4f}")

## Reconstruction quality

Unlike the synthetic blobs notebook, there's no hand-designed "shape factor" here to check recovery of -- MNIST digits already have real, richly structured within-class variation (slant, stroke width, size, closure of the loop in `0`, curvature in `1`). The relevant oracle is the same idea as before, adapted to images: predict nothing but the *class-average image* (the pixel-wise mean of all `0`s, or all `1`s) -- no per-image information at all. If the model's reconstructions do better than that, $z$ is encoding real, image-specific style, not just "which digit".

In [ ]:
with torch.no_grad():
    x_hat_final = decoder(z_final)
    x_val_hat_final = decoder(z_val_final)

x_hat_pca = pca_raw.transform(x_hat_final.reshape(N, -1).numpy())

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, data, title in [(axes[0], x_train_pca, "Original x_train (PCA)"), (axes[1], x_hat_pca, "Reconstructed decoder(z_train) (PCA)")]:
    ax.scatter(data[:, 0], data[:, 1], c=y_train.numpy(), cmap='coolwarm', s=15, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
plt.tight_layout()
plt.show()

class_mean_a = x_train[y_train == 0].mean(dim=0, keepdim=True)
class_mean_b = x_train[y_train == 1].mean(dim=0, keepdim=True)
class_means = torch.cat([class_mean_a, class_mean_b], dim=0)

mse_model = F.mse_loss(x_hat_final, x_train).item()
mse_oracle = F.mse_loss(class_means[y_train], x_train).item()
mse_val_model = F.mse_loss(x_val_hat_final, x_val).item()
mse_val_oracle = F.mse_loss(class_means[y_val], x_val).item()

print(f"Train -- MSE, class-average-image oracle: {mse_oracle:.5f}")
print(f"Train -- MSE, model reconstruction:        {mse_model:.5f}")
print(f"Val   -- MSE, class-average-image oracle: {mse_val_oracle:.5f}")
print(f"Val   -- MSE, model reconstruction:        {mse_val_model:.5f}")

### A closer look: actual reconstructed digits

The PCA scatter above shows population-level structure, but the most direct check for real images is just looking at them: original digits next to their own reconstructions, for a handful of examples from each class.

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(13, 3))
show_idx_a = (y_train == 0).nonzero().flatten()[:5]
show_idx_b = (y_train == 1).nonzero().flatten()[:5]
show_idx = torch.cat([show_idx_a, show_idx_b])
for col, i in enumerate(show_idx):
    axes[0, col].imshow(x_train[i, 0], cmap='gray', vmin=0, vmax=1); axes[0, col].axis('off')
    axes[1, col].imshow(x_hat_final[i, 0], cmap='gray', vmin=0, vmax=1); axes[1, col].axis('off')
axes[0, 0].set_title('original', loc='left', fontsize=9)
axes[1, 0].set_title('reconstruction', loc='left', fontsize=9)
plt.tight_layout()
plt.show()

## Inference on held-out data (Algorithm 2)

Same idea as `dgd_test_inference.ipynb` and the validation phase above: freeze the trained decoder $f_\theta$ and GMM, and optimize only the latents of genuinely unseen data -- the test split carved out at the very top of the notebook:

$$
\hat z_i = \arg\min_{z} \; \|f_\theta(\tilde z) - x_i\|_2^2 \;-\; \lambda \log p_{\text{GMM}}(\tilde z), \qquad m = 1, \ldots, M
$$

starting from the same zero-init used for $Z_0$, with a reconstruction-only warm-up ($M_0$ steps) before the GMM term is added, and the same noise schedule (mapped onto step index $m$ instead of epoch).

In [ ]:
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

test_rep = RepresentationLayer(dim=dim_z, n_samples=N_test, dist='zeros', dist_params={}, device=device)
test_optimizer = torch.optim.AdamW(test_rep.parameters(), lr=0.1, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0)
M = 100
M0 = first_epoch_gmm
test_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(test_optimizer, T_max=M, eta_min=0.01)

gmm_means_frozen = gmm.means_.detach().cpu().numpy()
gmm_vars_frozen = gmm.covariances_.detach().cpu().numpy()

print(f"Test set: {N_test} digits (never used in training or validation), optimizing for {M} steps (warm-up: {M0})")

In [ ]:
step_history = {'loss': [], 'recon': [], 'gmm': [], 'noise_scale': [], 'noise_realized': []}
test_frames = []

for m in range(1, M + 1):
    test_optimizer.zero_grad()
    noise_scale_m = cosine_noise_schedule(m, M, latent_noise_start, latent_noise_end)

    z_clean = test_rep()
    noise_m = torch.randn_like(z_clean) * noise_scale_m if noise_scale_m > 0 else torch.zeros_like(z_clean)
    z = z_clean + noise_m

    y_hat = decoder(z)
    recon_loss = F.mse_loss(y_hat, x_test, reduction='sum')

    if m >= M0:
        gmm_error = -lambda_gmm * gmm.score_samples(z).sum()
        loss = recon_loss + gmm_error
    else:
        gmm_error = torch.tensor(0.0)
        loss = recon_loss

    loss.backward()
    test_optimizer.step()
    test_scheduler.step()

    step_history['loss'].append(loss.item() / N_test)
    step_history['recon'].append(recon_loss.item() / N_test)
    step_history['gmm'].append(gmm_error.item() / N_test)
    step_history['noise_scale'].append(noise_scale_m)
    step_history['noise_realized'].append(noise_m.norm(dim=1).mean().item())

    with torch.no_grad():
        z_clean_snap = test_rep().detach()
        x_hat_clean = decoder(z_clean_snap)
    test_frames.append({
        'step': m,
        'z': z_clean_snap.clone().numpy(),
        'z_noised': z.detach().clone().numpy(),
        'x_hat_pca': pca_raw.transform(x_hat_clean.reshape(N_test, -1).numpy()),
        'means': gmm_means_frozen if m >= M0 else None,
        'vars': gmm_vars_frozen if m >= M0 else None,
    })

    if m % max(1, M // 10) == 0 or m == M:
        gmm_str = f"{step_history['gmm'][-1]:.4f}" if m >= M0 else "0.0000"
        print(f"Step {m}/{M} [LR: Rep={test_optimizer.param_groups[0]['lr']:.2e}, Noise={noise_scale_m:.4f}]")
        print(f"       - Loss: {step_history['loss'][-1]:.4f}, Recon: {step_history['recon'][-1]:.4f}, GMM: {gmm_str}")

print("Test optimization complete.")
print(f"Realized noise displacement, step 1 -> {M}: {step_history['noise_realized'][0]:.4f} -> {step_history['noise_realized'][-1]:.4f}")

### Watching inference converge

Same three-panel animation, same helper function used for training and validation above, now for the fully held-out test split.

In [ ]:
build_three_panel_gif(
    test_frames, x_test_pca, y_test.numpy(), 'toy_dgd_mnist_inference.gif',
    step_total=M, frame_stride=1, duration=250,
    left_title="True x_test (PCA)", mid_title="Latent z_test", right_title="Reconstruction (PCA)",
)

![Inference animation: true test digits (left), latent z_test with a displacement line per point converging against the frozen GMM (middle), reconstruction (right)](toy_dgd_mnist_inference.gif)

In [ ]:
z_test_final = test_rep().detach()
z_test_pred = gmm.predict(z_test_final)
test_ami = cluster_metrics.adjusted_mutual_info_score(y_test, z_test_pred)
test_ari = cluster_metrics.adjusted_rand_score(y_test, z_test_pred)
print(f"Held-out test data vs. the (frozen, trained) GMM: AMI={test_ami:.4f}, ARI={test_ari:.4f}")

fig, ax = plt.subplots(figsize=(6, 6))
plot_gmm(
    z_test_final, gmm=gmm,
    color_by_cluster=True, true_labels=y_test, match_labels_to_true=True,
    show_ellipses=True, ellipse_std_devs=[1, 2, 3],
    title="Held-out test latents against the trained (frozen) GMM",
    xlabel="z[0]", ylabel="z[1]",
    ax=ax,
)
plt.tight_layout()
plt.show()

### Reconstruction quality on held-out data

Same class-average-image oracle as the training-side reconstruction, now for `x_test`, plus the same original-vs-reconstruction image grid.

In [ ]:
with torch.no_grad():
    x_test_hat = decoder(z_test_final)

mse_test_model = F.mse_loss(x_test_hat, x_test).item()
mse_test_oracle = F.mse_loss(class_means[y_test], x_test).item()

print(f"MSE, class-average-image oracle: {mse_test_oracle:.5f}")
print(f"MSE, model reconstruction:        {mse_test_model:.5f}")

fig, axes = plt.subplots(2, 10, figsize=(13, 3))
show_idx_a = (y_test == 0).nonzero().flatten()[:5]
show_idx_b = (y_test == 1).nonzero().flatten()[:5]
show_idx = torch.cat([show_idx_a, show_idx_b])
for col, i in enumerate(show_idx):
    axes[0, col].imshow(x_test[i, 0], cmap='gray', vmin=0, vmax=1); axes[0, col].axis('off')
    axes[1, col].imshow(x_test_hat[i, 0], cmap='gray', vmin=0, vmax=1); axes[1, col].axis('off')
axes[0, 0].set_title('original (test)', loc='left', fontsize=9)
axes[1, 0].set_title('reconstruction', loc='left', fontsize=9)
plt.tight_layout()
plt.show()

## Takeaway

Same objective, same optimization recipe, same evaluation flow as `toy_dgd_blobs.ipynb` and the main pipeline -- but this time with real image data and the exact `ConvDecoder`/`config.yaml` architecture the FashionMNIST pipeline uses, not a hand-designed synthetic distribution and a plain MLP. The result: reconstructions visibly capture real per-digit style (slant, stroke width) rather than collapsing to a class-average image, using the same 2D latent, the same noise schedule, and the same GMM prior strength that suppressed all fine structure on the synthetic blobs. That contrast is itself informative -- it wasn't the *mechanism* that failed on the synthetic data, it was a mismatch between FashionMNIST-calibrated regularization strength and an information-poor (i.i.d.-noise) toy distribution. Real image data, even at this tiny two-class scale, has enough structure to survive it.